In [5]:
import sys
from pathlib import Path

# Add the parent directory to the system path
sys.path.append(str(Path.cwd().parent.resolve()))

In [6]:
# Load Libraries
import numpy as np
import pandas as pd
from datetime import datetime
import plotly.express as px
from matplotlib import pyplot as plt
from prettytable import PrettyTable
from sklearn.linear_model import LinearRegression

In [7]:
# Data Path
path = str(Path.cwd().parent.resolve())
dma_path = path + "/data/DMA_X.csv"

In [9]:
# Load Data
dma_flow = pd.read_csv(dma_path, index_col="TimeStamp", date_format="%d/%m/%Y %H:%M")
dma_flow["Flow"] = dma_flow["Flow"].ewm(alpha=0.75, adjust=False).mean()
dma_flow["Weekday"] = dma_flow.index.dayofweek

# Curate Flows between 00:00 - 6:00 and Visualize
hourly_dma_flow = dma_flow.resample("1h").mean()
px.line(hourly_dma_flow, x=hourly_dma_flow.index, y=["Flow"])
plt.show()

In [10]:
# Curate Leak and Non Leak Period
non_leak_data = hourly_dma_flow.loc["2022-04-04 00:00:00":"2022-04-10 23:59:00"]
leak_data = hourly_dma_flow.loc["2022-06-06 00:00:00":"2022-06-12 23:55:00"]

# Curate Corresponding Leak and Non Leak Index
day_of_week = 1
nlp_idx = non_leak_data["Weekday"].isin([day_of_week]).values
lp_idx = leak_data["Weekday"].isin([day_of_week]).values

# Sorted Data on Non-Leak Period
sorted_idx = np.argsort(non_leak_data["Flow"][nlp_idx].values.flatten())
non_leak_period = non_leak_data["Flow"][nlp_idx].values[sorted_idx].reshape(-1, 1)
leak_period = leak_data["Flow"][lp_idx].values[sorted_idx].reshape(-1, 1)

# Fit Linear Model
model = LinearRegression()
model.fit(non_leak_period, leak_period)

# Ground Truth Leak Magnitude
true_leak_magnitude = 3.0

# Leak Magnitude Estimate
estimated_leak_magnitude = np.round(model.intercept_[0], 2)
percentage_deviation = (
    np.abs(estimated_leak_magnitude - true_leak_magnitude) / true_leak_magnitude
) * 100

# Print Table of Scores
table_entries = [[true_leak_magnitude, estimated_leak_magnitude, np.round(percentage_deviation, 2)]]
scores_table = PrettyTable(
    ["True Leak Magnitude", "Estimated Leak Magnitude", "Percentage Deviation (%)"]
)
scores_table.add_rows(table_entries)
print(scores_table)

+---------------------+--------------------------+--------------------------+
| True Leak Magnitude | Estimated Leak Magnitude | Percentage Deviation (%) |
+---------------------+--------------------------+--------------------------+
|         3.0         |           3.13           |           4.33           |
+---------------------+--------------------------+--------------------------+
